In [3]:
import requests
import pandas as pd
from io import StringIO
from pathlib import Path
import os

In [1]:
import getpass

import os
from dotenv import load_dotenv

load_dotenv()

MAP_KEY = os.getenv("NASA_FIRMS_MAP_KEY")

if not MAP_KEY:
    raise ValueError("NASA FIRMS API key not found!")

In [2]:
url = (
    "https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
    f"{MAP_KEY}/VIIRS_SNPP_NRT/world/1"
)
print("URL created successfully")

URL created successfully


In [4]:
response = requests.get(url, timeout=60)
response.raise_for_status()

df = pd.read_csv(StringIO(response.text))

print("Number of records:", len(df))
display(df.tail())

Number of records: 31627


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
31622,50.39798,-118.84299,357.25,0.4,0.6,2026-10-30,1946,N,VIIRS,l,2.0URT,276.82,31.75,D
31623,50.40236,-118.84916,357.42,0.4,0.6,2026-10-30,1946,N,VIIRS,l,2.0URT,274.90,24.34,D
31624,50.40543,-118.85947,346.57,0.4,0.6,2026-10-30,1946,N,VIIRS,n,2.0URT,272.20,28.11,D
31625,50.40688,-118.85485,356.76,0.4,0.6,2026-10-30,1946,N,VIIRS,l,2.0URT,273.54,24.34,D
31626,50.41452,-115.62442,327.15,0.6,0.5,2026-10-30,1946,N,VIIRS,n,2.0URT,266.54,3.70,D


In [6]:
print("Dataset shape:", df.shape)

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())



Dataset shape: (49481, 14)

Data types:
latitude      float64
longitude     float64
bright_ti4    float64
scan          float64
track         float64
acq_date          str
acq_time        int64
satellite         str
instrument        str
confidence        str
version           str
bright_ti5    float64
frp           float64
daynight          str
dtype: object

Missing values:
latitude      0
longitude     0
bright_ti4    0
scan          0
track         0
acq_date      0
acq_time      0
satellite     0
instrument    0
confidence    0
version       0
bright_ti5    0
frp           0
daynight      0
dtype: int64

Duplicate rows: 0


In [7]:

# 1. Check data types
print("Data types:")
display(df.dtypes)

# 2. Check latitude and longitude ranges
print("Invalid latitude:", (~df["latitude"].between(-90, 90)).sum())
print("Invalid longitude:", (~df["longitude"].between(-180, 180)).sum())

# 3. Check invalid numeric values
numeric_cols = [
    "latitude", "longitude", "bright_ti4",
    "bright_ti5", "scan", "track", "frp"
]

print("\nNegative values in numeric columns:")
display((df[numeric_cols].lt(0)).sum())

# 4. Check date and time values
print("\nAcquisition dates:")
print(df["acq_date"].min(), "to", df["acq_date"].max())

print("\nSatellite distribution:")
display(df["satellite"].value_counts())

print("\nDay/Night distribution:")
display(df["daynight"].value_counts())


Data types:


latitude      float64
longitude     float64
bright_ti4    float64
scan          float64
track         float64
acq_date          str
acq_time        int64
satellite         str
instrument        str
confidence        str
version           str
bright_ti5    float64
frp           float64
daynight          str
dtype: object

Invalid latitude: 0
Invalid longitude: 0

Negative values in numeric columns:


latitude      27959
longitude     10432
bright_ti4        0
bright_ti5        0
scan              0
track             0
frp               0
dtype: int64


Acquisition dates:
2026-10-28 to 2026-10-28

Satellite distribution:


satellite
N    49481
Name: count, dtype: int64


Day/Night distribution:


daynight
D    43177
N     6304
Name: count, dtype: int64

In [8]:

# Create a separate working copy
df_clean = df.copy()

# Convert date and time into proper data types
df_clean["acq_date"] = pd.to_datetime(
    df_clean["acq_date"], errors="coerce"
)

df_clean["acq_time"] = (
    df_clean["acq_time"]
    .astype("string")
    .str.zfill(4)
)

# Validate the time format (HHMM)
valid_time = df_clean["acq_time"].str.fullmatch(
    r"(?:[01]\d|2[0-3])[0-5]\d"
)

print("Invalid dates:", df_clean["acq_date"].isna().sum())
print("Invalid times:", (~valid_time).sum())

# Check the categorical columns
for col in ["confidence", "version", "satellite",
            "instrument", "daynight"]:
    print(f"\n{col}:")
    print(df_clean[col].value_counts(dropna=False))


Invalid dates: 0
Invalid times: 0

confidence:
confidence
n    38546
l     8231
h     2704
Name: count, dtype: int64

version:
version
2.0NRT    48779
2.0URT      702
Name: count, dtype: int64

satellite:
satellite
N    49481
Name: count, dtype: int64

instrument:
instrument
VIIRS    49481
Name: count, dtype: int64

daynight:
daynight
D    43177
N     6304
Name: count, dtype: int64


In [9]:

# Check duplicated detections using a subset of columns
check_cols = [
    "latitude", "longitude",
    "acq_date", "acq_time",
    "satellite", "instrument"
]

print(
    "Potential duplicate detections:",
    df_clean.duplicated(subset=check_cols).sum()
)

# Check missing and invalid categorical values
print("\nUnexpected day/night values:")
print(~df_clean["daynight"].isin(["D", "N"]).value_counts())

print("\nNegative FRP values:", (df_clean["frp"] < 0).sum())
print("Non-positive brightness TI4:", (df_clean["bright_ti4"] <= 0).sum())

# Inspect records by version and confidence
display(
    df_clean.groupby(["version", "confidence"], dropna=False)
    .size()
    .rename("records")
    .reset_index()
)


Potential duplicate detections: 0

Unexpected day/night values:
daynight
True   -49482
Name: count, dtype: int64

Negative FRP values: 0
Non-positive brightness TI4: 0


,version,confidence,records
0,2.0NRT,h,2661
1,2.0NRT,l,8196
2,2.0NRT,n,37922
3,2.0URT,h,43
4,2.0URT,l,35
5,2.0URT,n,624


In [10]:

valid_daynight = df_clean["daynight"].isin(["D", "N"])

print("Invalid day/night records:", (~valid_daynight).sum())

print("Existing values:")
print(df_clean["daynight"].value_counts(dropna=False))


Invalid day/night records: 0
Existing values:
daynight
D    43177
N     6304
Name: count, dtype: int64


In [11]:
from pathlib import Path

raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(raw_dir / "fires_raw.csv", index=False)

print("Raw data saved:", len(df), "records")

Raw data saved: 49481 records


In [12]:
import pandas as pd

df_clean = df.copy()

# Convert date
df_clean["acq_date"] = pd.to_datetime(
    df_clean["acq_date"],
    errors="raise"
).dt.date

# Convert HHMM to a proper UTC timestamp
time_str = df_clean["acq_time"].astype(str).str.zfill(4)

df_clean["acq_timestamp_utc"] = pd.to_datetime(
    df_clean["acq_date"].astype(str) + " " +
    time_str.str[:2] + ":" + time_str.str[2:] + ":00",
    utc=True,
    errors="raise"
)

# Review the result
display(df_clean[
    ["acq_date", "acq_time", "acq_timestamp_utc"]
].head())

,acq_date,acq_time,acq_timestamp_utc
0,2026-10-28,20,2026-10-28 00:20:00+00:00
1,2026-10-28,20,2026-10-28 00:20:00+00:00
2,2026-10-28,20,2026-10-28 00:20:00+00:00
3,2026-10-28,20,2026-10-28 00:20:00+00:00
4,2026-10-28,20,2026-10-28 00:20:00+00:00


In [13]:

from pathlib import Path

processed_dir = Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

output_file = processed_dir / "fires_clean.csv"

df_clean.to_csv(output_file, index=False)

print("Processed data saved successfully!")
print("Rows:", len(df_clean))
print("Columns:", len(df_clean.columns))
print("File:", output_file)


Processed data saved successfully!
Rows: 49481
Columns: 15
File: data\processed\fires_clean.csv
